In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Suppresses interactive GUI popups to prevent environment hangs
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

print("🚀 Launching Natural Language Processing Pipeline...")

# 1. LOAD AND PROFILE TEXT DATASET
data_file = 'customer_support_text_classification.csv'
if not os.path.exists(data_file):
    print(f"❌ Error: Cannot find '{data_file}' in the active directory.")
    raise FileNotFoundError

df = pd.read_csv(data_file)
print(f"Dataset Shape: {df.shape}")
print("\nSentiment Label Class Balance:")
print(df['sentiment_label'].value_counts())

# 2. LEXICAL CLEANING PIPELINE ENGINE
def clean_support_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  # Case normalization
    text = re.sub(r'ticket number is \d+', '', text)  # Strip specific ID tokens
    text = re.sub(r'[^\w\s]', '', text)  # Punctuation clearing
    text = re.sub(r'\s+', ' ', text).strip()  # Whitespace compression
    return text

print("\nFormatting string vectors and isolating text bodies...")
df['cleaned_message'] = df['customer_message'].apply(clean_support_text)

# Separate input strings and class arrays
X = df['cleaned_message']
y = df['sentiment_label']

# Stratified division split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. TEXT VECTORIZATION MATRIX STRATEGY (TF-IDF)
# Restricting to top 1,000 structural unigrams/bigrams to manage document-term sparsity
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Train Matrix Shape: {X_train_vec.shape}")
print(f"Validation Matrix Shape: {X_test_vec.shape}")

# 4. BASELINE MODEL TRAINING & METRICS LOGGING
model = LogisticRegression(C=1.0, max_iter=200, random_state=42)
model.fit(X_train_vec, y_train)

# Evaluate predictions
y_preds = model.predict(X_test_vec)
unique_labels = sorted(df['sentiment_label'].unique())
cm = confusion_matrix(y_test, y_preds, labels=unique_labels)

# 5. DISK OUTPUT COMPILATION
os.makedirs('results', exist_ok=True)

# Generate textual verification log
report_dict = classification_report(y_test, y_preds, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv('results/model_comparison_table.csv', index=True)

# Render and save evaluation graphics silently
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=unique_labels, yticklabels=unique_labels, ax=ax, cbar=False)
ax.set_title('Sentiment Classification Confusion Matrix')
ax.set_xlabel('Predicted Sentiment State')
ax.set_ylabel('True Sentiment Label')

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=300)
plt.close(fig)

print("\n✅ Part 3 Execution Complete!")
print("-> Saved accuracy classifications to: results/model_comparison_table.csv")
print("-> Saved visual validation heatmaps to: results/evaluation_outputs.png")

🚀 Launching Natural Language Processing Pipeline...
Dataset Shape: (1500, 6)

Sentiment Label Class Balance:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64

Formatting string vectors and isolating text bodies...
Train Matrix Shape: (1200, 353)
Validation Matrix Shape: (300, 353)

✅ Part 3 Execution Complete!
-> Saved accuracy classifications to: results/model_comparison_table.csv
-> Saved visual validation heatmaps to: results/evaluation_outputs.png
